In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)

# Load dataset
df = pd.read_csv("heart.csv")

# Identify target column
target_col = None
for candidate in ["target", "heart_disease", "heart disease", "num"]:
    if candidate in df.columns:
        target_col = candidate
        break
if target_col is None:
    target_col = df.columns[-1]

# Clean dataset
print("Shape:", df.shape)
print("Missing values per column:")
print(df.isna().sum())
df = df.copy()
if df.isna().sum().sum() > 0:
    df.fillna(df.median(numeric_only=True), inplace=True)

# Convert target to binary if needed
df[target_col] = df[target_col].astype(int)

# Basic EDA
print("\nTarget distribution:")
print(df[target_col].value_counts(normalize=True))

plt.figure(figsize=(6, 4))
sns.countplot(x=target_col, data=df)
plt.title("Heart Disease Target Distribution")
plt.xlabel("Target")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(12, 10))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Matrix")
plt.show()

# Features and target
X = df.drop(columns=[target_col])
X = pd.get_dummies(X, drop_first=True)
y = df[target_col]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.25, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train models
lr = LogisticRegression(max_iter=1000, solver="liblinear")
lr.fit(X_train_scaled, y_train)

dt = DecisionTreeClassifier(random_state=42, max_depth=5)
dt.fit(X_train, y_train)

# Evaluation helper
def evaluate_model(model, X_eval, y_eval, name):
    y_pred = model.predict(X_eval)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_eval)[:, 1]
    else:
        y_score = model.decision_function(X_eval)
    acc = accuracy_score(y_eval, y_pred)
    roc = roc_auc_score(y_eval, y_score)
    cm = confusion_matrix(y_eval, y_pred)
    print(f"\n{name} evaluation")
    print("Accuracy:", round(acc, 4))
    print("ROC AUC:", round(roc, 4))
    print("Confusion matrix:\n", cm)
    print("Classification report:")
    print(classification_report(y_eval, y_pred, digits=4))
    return y_score, cm

lr_scores, lr_cm = evaluate_model(lr, X_test_scaled, y_test, "Logistic Regression")
dt_scores, dt_cm = evaluate_model(dt, X_test, y_test, "Decision Tree")

# ROC curve
plt.figure(figsize=(8, 6))
for name, scores in [("Logistic Regression", lr_scores), ("Decision Tree", dt_scores)]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, scores):.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.show()

# Confusion matrix for selected model (Logistic Regression)
plt.figure(figsize=(5, 4))
sns.heatmap(lr_cm, annot=True, fmt="d", cmap="Blues")
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Feature importance
feature_names = X.columns

lr_importance = pd.Series(
    np.abs(lr.coef_[0]), index=feature_names
).sort_values(ascending=False)
dt_importance = pd.Series(dt.feature_importances_, index=feature_names).sort_values(ascending=False)

print("\nTop 10 logistic regression feature importances:")
print(lr_importance.head(10))

print("\nTop 10 decision tree feature importances:")
print(dt_importance.head(10))

plt.figure(figsize=(8, 5))
lr_importance.head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Logistic Regression Feature Importances")
plt.xlabel("Absolute Coefficient Value")
plt.show()